## 선택 · 심화 문제 1. `max_length` 후보 절단 감사

### 문제 배경

최대 길이는 관습적으로 정하지 않고 원본 token 길이 분포와 절단되는 실제 예시를 근거로 선택합니다.

### 시작 코드

```python
texts = ["짧은 제목", "한국어 뉴스 제목의 토큰 길이를 측정합니다", "매우 긴 제목 " * 20]

def audit_max_lengths(texts, candidates=(8, 16, 32)):
    raise NotImplementedError
```

### 수행 요구사항

1. Truncation 없이 special token을 포함한 원래 길이를 측정하세요.
2. 후보별 `truncated_count/rate`, 보존 token 비율을 계산하세요.
3. 절단되는 행 index를 반환하세요.
4. 절단 비율이 가장 낮고, 동률이면 작은 후보를 선택하세요.

### 제출 결과

- 원래 길이와 후보별 표
- 선택한 max length와 수치 기반 이유
- 절단되는 실제 입력의 한계
- `심화 문제 1 자동 검증: PASS`

### 자동 검증

```python
report = audit_max_lengths(texts)
assert set(report["candidates"]) == {8, 16, 32}
assert report["candidates"][8]["truncated_count"] >= report["candidates"][32]["truncated_count"]
assert report["selected"] == 16
print("심화 문제 1 자동 검증: PASS")
```
    ```
    
  **상세 해설** · 이 정책은 절단 건수만 최소화하므로 GPU 메모리 예산을 반영하지 않습니다. 실제 선택은 validation 성능, throughput, peak memory와 절단된 예시의 정보 손실을 함께 봅니다.
    
   **자주 하는 실수**
    
    - 이미 truncation한 길이로 절단 비율을 계산해 항상 0%로 만듭니다.
    - 문자 길이와 token 길이를 혼동합니다.
    - 평균만 보고 긴 꼬리와 실제 절단 사례를 보지 않습니다.

---

In [9]:
from transformers import AutoTokenizer

MODEL_ID = "monologg/koelectra-small-v3-discriminator"
texts = ["짧은 제목", "한국어 뉴스 제목의 토큰 길이를 측정합니다", "매우 긴 제목 " * 20]

def audit_max_lengths(texts, candidates=(8, 16, 32)):
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, local_files_only=True)
    # 아직 자르지 않은 원본 token 길이를 측정해야 절단률이 왜곡되지 않습니다.
    lengths = [len(tokenizer(text, add_special_tokens=True, truncation=False)["input_ids"])
               for text in texts]
    total_tokens = sum(lengths)
    table = {}
    for candidate in candidates:
        truncated_rows = [i for i, length in enumerate(lengths) if length > candidate]
        kept_tokens = sum(min(length, candidate) for length in lengths)
        table[candidate] = {
            "truncated_count": len(truncated_rows),
            "truncated_rate": len(truncated_rows) / len(texts),
            "retained_token_rate": kept_tokens / total_tokens,
            "truncated_rows": truncated_rows,
        }
    # 절단 건수 최소, 동률이면 더 작은 길이 순으로 선택합니다.
    # 절단 행 수를 먼저 최소화하고 동률이면 더 짧은 후보를 선택합니다.
    selected = min(candidates, key=lambda c: (table[c]["truncated_count"], c))
    return {"original_lengths": lengths, "candidates": table, "selected": selected}

report = audit_max_lengths(texts)
print(report)
assert set(report["candidates"]) == {8, 16, 32}
assert report["candidates"][8]["truncated_count"] >= report["candidates"][32]["truncated_count"]
assert report["selected"] == 16
print("심화 문제 1 자동 검증: PASS")

{'original_lengths': [5, 11, 62], 'candidates': {8: {'truncated_count': 2, 'truncated_rate': 0.6666666666666666, 'retained_token_rate': 0.2692307692307692, 'truncated_rows': [1, 2]}, 16: {'truncated_count': 1, 'truncated_rate': 0.3333333333333333, 'retained_token_rate': 0.41025641025641024, 'truncated_rows': [2]}, 32: {'truncated_count': 1, 'truncated_rate': 0.3333333333333333, 'retained_token_rate': 0.6153846153846154, 'truncated_rows': [2]}}, 'selected': 16}
심화 문제 1 자동 검증: PASS
